# Kaggleコンペ: Biohub - Cell Tracking During Development
## 解説付き学習用notebook（写経 + 解説）

- **コンペ**: [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)（ゼブラフィッシュ胚発生過程の3D顕微鏡動画から、細胞を検出し時間を通して追跡するコンペ。研究コード（Research Code）コンペで賞金 $60,000、開催中）
- **元notebook**: [Cell Tracking Getting Started w/ Nearest Neighbor](https://www.kaggle.com/code/inversion/cell-tracking-getting-started-w-nearest-neighbor) by **inversion**（主催者（Kaggle公式）によるベースラインnotebook、349 upvotes、Pinned Notebook）
- **タスク概要**: 4次元（時間×Z×Y×X）の3D顕微鏡ボリュームデータから、各時刻ごとに細胞の位置を検出し、フレーム間で同じ細胞を追跡してその軌跡（トラック）を再構築するタスクです。
- **手法の概要**: 各フレームを間引いて平滑化した後、明るさの上位パーセンタイルを閾値として2値化し、連結成分ラベリングで細胞候補を検出します。フレーム間の対応付けは、重心間のユークリッド距離をコストとしたハンガリアン法（`scipy.optimize.linear_sum_assignment`）で解き、最近傍同士を1対1で結びつけるシンプルな「検出→追跡」パイプラインです。

> **注記**: これは学習目的で作成した解説付きの写しです。コード自体は原著者（Kaggle運営）のものをほぼそのまま保持しており、大きな改変はしていません。実際には実行しておらず、出力結果は含まれていません。各コードセルの直前に、初心者向けの「何をしているか」「なぜそうするのか」の解説を日本語で追加しています。


**何をしているか**: 必要なライブラリを読み込んでいます。`blosc2`は3D顕微鏡画像（Zarr形式で圧縮保存された大容量データ）を解凍するための圧縮ライブラリ、`scipy.ndimage`は画像処理（平滑化・ラベリング）、`scipy.optimize.linear_sum_assignment`は「割り当て問題」を解く関数（ハンガリアン法）です。

**なぜそうするのか**: このコンペのデータは3D+時間（4次元）の顕微鏡動画で、ファイルサイズが非常に大きいため専用の圧縮形式（Zarr/blosc2）で配布されています。それを読み解くための専用ツールが必要になります。

In [ ]:
import json
import os

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import label, uniform_filter
from scipy.optimize import linear_sum_assignment

**何をしているか**: データが置かれているフォルダのパスと、画像処理のためのパラメータをまとめて定義しています。`SCALE`は各軸（Z/Y/X）1ボクセルあたりの実際の長さ（マイクロメートル）、`DOWNSAMPLE=4`は画像を1/4に間引いて処理を軽くする設定、`PERCENTILE=90`は明るさの上位10%を「細胞」とみなす閾値、`MAX_LINK_DISTANCE`は前後のフレームで「同じ細胞」とみなす最大移動距離です。

**なぜそうするのか**: 3D顕微鏡データは非常に大きく高解像度なため、そのまま処理すると時間がかかりすぎます。ダウンサンプリング（間引き）で計算量を減らしつつ、閾値や距離といった調整可能なパラメータを最初にまとめておくことで、後から手軽にチューニングできるようにしています。実際の細胞の大きさや動く速さに対して現実的な値（マイクロメートル単位）を使っている点が「物理的な意味を考えた」工夫です。

In [ ]:
TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X µm/voxel
DOWNSAMPLE = 4
PERCENTILE = 90
MAX_LINK_DISTANCE = 15.0  # µm

**何をしているか**: このノートブックの本体です。各動画（フォルダ）について、時刻ごとに3D画像を読み込み、（1）ダウンサンプリングして平滑化し、（2）明るい部分90%タイルを閾値として2値化（細胞か背景かを判定）し、（3）`scipy.ndimage.label`で隣接する明るいボクセルの塊（連結成分）ごとに番号を振って「1つの細胞」として重心座標を計算します。さらに、直前の時刻の細胞群と今の時刻の細胞群の重心間の距離行列を作り、`linear_sum_assignment`（ハンガリアン法）で最も辻褄の合う1対1の対応付け（どの細胞がどの細胞に移動したか）を求め、距離が近すぎる（`MAX_LINK_DISTANCE`以下の）ペアだけを「同じ細胞の追跡」として記録しています。

**なぜそうするのか**: これは典型的な「検出→追跡」の2段階アプローチです。まず各フレームで細胞の位置を検出し（Detection）、次にフレーム間で同じ細胞同士を線でつなぐ（Tracking/Linking）ことで、時間を通した細胞の動きを再構築します。ハンガリアン法を使うのは、単純に一番近い細胞同士を貪欲に繋ぐと「取り合い」が起きて矛盾が生じることがあるため、全体最適な1対1の対応を数学的に求めるためです。細胞追跡・物体追跡タスクの定番の考え方（検出はセグメンテーション、追跡は割り当て問題として解く）を、シンプルな道具だけで実装した好例です。

In [ ]:
test_folder_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')
)

all_rows = []
for folder_name in test_folder_names:
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')

    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        arr_meta = json.load(f)
    shape = tuple(arr_meta['shape'])  # (T, Z, Y, X)
    dtype = np.dtype(arr_meta['data_type'])
    n_t = shape[0]

    prev_centroids = {}
    node_id_counter = 1

    for t in range(n_t):
        chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as f:
            compressed = f.read()
        decompressed = blosc2.decompress(compressed)
        vol = np.frombuffer(decompressed, dtype=dtype).reshape(shape[1:])  # (Z, Y, X)

        ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]

        smoothed = uniform_filter(ds.astype(np.float32), size=3)
        threshold = np.percentile(smoothed, PERCENTILE)
        binary = smoothed > threshold

        labeled, n_features = label(binary)

        centroids = {}
        for comp_id in range(1, n_features + 1):
            coords = np.argwhere(labeled == comp_id)
            centroid = coords.mean(axis=0) * DOWNSAMPLE
            nid = node_id_counter
            node_id_counter += 1
            centroids[nid] = (int(round(centroid[0])), int(round(centroid[1])), int(round(centroid[2])))
            all_rows.append({
                'dataset': folder_name,
                'row_type': 'node',
                'node_id': nid,
                't': t,
                'z': centroids[nid][0],
                'y': centroids[nid][1],
                'x': centroids[nid][2],
                'source_id': -1,
                'target_id': -1,
            })

        if prev_centroids:
            prev_ids = list(prev_centroids.keys())
            curr_ids = list(centroids.keys())
            if prev_ids and curr_ids:
                prev_coords = np.array([prev_centroids[pid] for pid in prev_ids], dtype=np.float64)
                curr_coords = np.array([centroids[cid] for cid in curr_ids], dtype=np.float64)
                prev_phys = prev_coords * SCALE
                curr_phys = curr_coords * SCALE
                dist = np.sqrt(((prev_phys[:, None] - curr_phys[None, :]) ** 2).sum(axis=2))
                row_ind, col_ind = linear_sum_assignment(dist)
                for ri, ci in zip(row_ind, col_ind):
                    if dist[ri, ci] <= MAX_LINK_DISTANCE:
                        all_rows.append({
                            'dataset': folder_name,
                            'row_type': 'edge',
                            'node_id': -1,
                            't': -1,
                            'z': -1,
                            'y': -1,
                            'x': -1,
                            'source_id': prev_ids[ri],
                            'target_id': curr_ids[ci],
                        })

        prev_centroids = centroids

    print(f'{folder_name}: {node_id_counter - 1} nodes')

**何をしているか**: これまで集めた「ノード（各時刻・各細胞の位置）」と「エッジ（時刻をまたぐ細胞同士のつながり）」の記録を1つの表にまとめ、`submission.csv`として書き出しています。

**なぜそうするのか**: Kaggleコンペではこの提出ファイルの形式（ノードとエッジをまとめたテーブル）がスコアリングの対象になります。検出と追跡の結果をコンペが指定するフォーマットに変換する、地味だが欠かせない最後のステップです。

In [ ]:
submission = pd.DataFrame(all_rows)
submission.index.name = 'id'
submission.to_csv('submission.csv')
print(f'Done. {len(submission)} rows written to submission.csv')